<a href="https://colab.research.google.com/github/runfish5/NLP/blob/main/Model_wiki_GENRE/EntityLinking_GENRE_colab_MinimalCode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. colab GENRE
at the timepoint 2022.Oct.27, this is a fully functional colab script to run Facebook AI's entity linker called 'GENRE' (https://github.com/facebookresearch/GENRE). This file can be used as an entry point to write a colab script that also includes training (but I haven't written that yet).

A 2nd approach that utilizes huggingface-transfromers package is also provided in one cell, but that code is redundant for the 1st approach.

In [ ]:
#@title 1.2. huggingface GENRE
# # %%capture
# !pip install transformers

# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# # OPTIONAL: load the prefix tree (trie), you need to additionally download
# # https://huggingface.co/facebook/genre-kilt/blob/main/trie.py and
# # https://huggingface.co/facebook/genre-kilt/blob/main/kilt_titles_trie_dict.pkl
# # import pickle
# # from trie import Trie
# # with open("kilt_titles_trie_dict.pkl", "rb") as f:
# #     trie = Trie.load_from_dict(pickle.load(f))

# tokenizer = AutoTokenizer.from_pretrained("facebook/genre-kilt")
# model = AutoModelForSeq2SeqLM.from_pretrained("facebook/genre-kilt").eval()

# sentences = ["Einstein was a German physicist."]

# outputs = model.generate(
#     **tokenizer(sentences, return_tensors="pt"),
#     num_beams=5,
#     num_return_sequences=5,
#     # OPTIONAL: use constrained beam search
#     # prefix_allowed_tokens_fn=lambda batch_id, sent: trie.get(sent.tolist()),
# )

# tokenizer.batch_decode(outputs, skip_special_tokens=True)

# sentences = [" Proteins are well known in the realm of [START] synthetic biology, where E. coli oftenis used to produce the catalyst for experiments associated with the INS gene .."]

# outputs = model.generate(
#     **tokenizer(sentences, return_tensors="pt"),
#     num_beams=5,
#     num_return_sequences=5,
#     # OPTIONAL: use constrained beam search
#     # prefix_allowed_tokens_fn=lambda batch_id, sent: trie.get(sent.tolist()),
# )

# tokenizer.batch_decode(outputs, skip_special_tokens=True)

## 1.3. download GENRE & configure

In [ ]:
#@title clone GENRE
# %%capture
%cd /content/
!git clone https://github.com/facebookresearch/GENRE

/content
Cloning into 'GENRE'...
remote: Enumerating objects: 457, done.
remote: Counting objects: 100% (173/173), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 457 (delta 114), reused 102 (delta 78), pack-reused 284
Receiving objects: 100% (457/457), 11.00 MiB | 25.36 MiB/s, done.
Resolving deltas: 100% (261/261), done.


In [ ]:
#@title clone & install fairseq
%%capture
!git clone --branch fixing_prefix_allowed_tokens_fn https://github.com/nicola-decao/fairseq
!pwd
%cd /content/fairseq
! pip install --editable .
#! pip install --editable ./

In [ ]:
'''this path.append must maybe be before runtime restart.'''
import sys
sys.path.append("/content/fairseq/")
sys.path.append("/content/GENRE")

<font color='red'> restart runtime was initiially necessary here, now not anymore...! </font>

In [ ]:
%%capture
!pip install jsonlines

In [ ]:
%cd /content/
!mkdir data
%cd data
### KILT prefix tree
!wget http://dl.fbaipublicfiles.com/GENRE/kilt_titles_trie_dict.pkl

/content
/content/data
--2022-11-02 06:17:52--  http://dl.fbaipublicfiles.com/GENRE/kilt_titles_trie_dict.pkl
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 104.22.75.142, 172.67.9.4, 104.22.74.142, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|104.22.75.142|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 215214973 (205M) [application/octet-stream]
Saving to: ‘kilt_titles_trie_dict.pkl’

kilt_titles_trie_di 100%[===================>] 205.24M  26.0MB/s    in 8.5s    

2022-11-02 06:18:01 (24.3 MB/s) - ‘kilt_titles_trie_dict.pkl’ saved [215214973/215214973]



 from GENRE/scripts_genre/download_all_models.sh #

In [ ]:
'''
This code sniped is missing in the README.md,
but can be found at 'GENRE/scripts_genre/download_all_models.sh'
'''
%cd /content/
!mkdir models
%cd models
!wget http://dl.fbaipublicfiles.com/GENRE/fairseq_entity_disambiguation_aidayago.tar.gz
!tar -zxvf fairseq_entity_disambiguation_aidayago.tar.gz

/content
/content/models
--2022-11-02 06:18:01--  http://dl.fbaipublicfiles.com/GENRE/fairseq_entity_disambiguation_aidayago.tar.gz
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 104.22.75.142, 172.67.9.4, 104.22.74.142, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|104.22.75.142|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1201544043 (1.1G) [application/gzip]
Saving to: ‘fairseq_entity_disambiguation_aidayago.tar.gz’

fairseq_entity_disa 100%[===================>]   1.12G  32.0MB/s    in 38s     

2022-11-02 06:18:39 (30.3 MB/s) - ‘fairseq_entity_disambiguation_aidayago.tar.gz’ saved [1201544043/1201544043]

fairseq_entity_disambiguation_aidayago/
fairseq_entity_disambiguation_aidayago/dict.source.txt
fairseq_entity_disambiguation_aidayago/dict.target.txt
fairseq_entity_disambiguation_aidayago/model.pt
fairseq_entity_disambiguation_aidayago/encoder.json
fairseq_entity_disambiguation_aidayago/vocab.bpe


In [ ]:
import pickle
from genre.trie import Trie
%cd /content
# load the prefix tree (trie)
with open("data/kilt_titles_trie_dict.pkl", "rb") as f:
    trie = Trie.load_from_dict(pickle.load(f))

/content


## 1.3 the model

In [ ]:
from genre.fairseq_model import GENRE
%cd /content/
model = GENRE.from_pretrained("/content/models/fairseq_entity_disambiguation_aidayago").eval()

# for huggingface/transformers
# from genre.hf_model import GENRE
# model = GENRE.from_pretrained("../models/hf_entity_disambiguation_aidayago").eval()

/content


1042301B [00:00, 3081465.33B/s]
456318B [00:00, 882396.44B/s]


In [ ]:
# myPhrase = ["Einstein was a [START_ENT] German [END_ENT] physicist."]

myPhrase = [' Proteins are well known in the realm of synthetic biology, where E. coli often is used to produce material for experiments associated with the INS gene.',
            'Experiments associated with the [START_ENT] INS gene [END_ENT] or in very special cases other sources.',
            'Insulin is a small molecule. experiments associated with the [START_ENT] INS gene [END_ENT] or in very special cases other sources.']
model.sample(
    sentences=myPhrase,
    prefix_allowed_tokens_fn=lambda batch_id, sent: trie.get(sent.tolist()),
)

/content/fairseq/fairseq/search.py:205: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values. To keep the current behavior, use torch.div(a, b, rounding_mode='trunc'), or for actual floor division, use torch.div(a, b, rounding_mode='floor').
  beams_buf = indices_buf // vocab_size
/content/fairseq/fairseq/sequence_generator.py:659: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values. To keep the current behavior, use torch.div(a, b, rounding_mode='trunc'), or for actual floor division, use torch.div(a, b, rounding_mode='floor').
  unfin_idx = idx // beam_size


[[{'text': 'Biosynthesis', 'score': tensor(-1.4936)},
  {'text': 'Sodium silicate', 'score': tensor(-1.5992)},
  {'text': 'Biopterin', 'score': tensor(-1.6096)},
  {'text': 'Sodium silicide', 'score': tensor(-1.9425)},
  {'text': 'Biopterin-dependent aromatic amino acid hydroxylase',
   'score': tensor(-2.4378)}],
 [{'text': 'Inositol trisphosphate', 'score': tensor(-0.1708)},
  {'text': 'Insulin-like growth factor', 'score': tensor(-0.5887)},
  {'text': 'Immunoglobulin E', 'score': tensor(-0.9796)},
  {'text': 'Immunoglobulin A', 'score': tensor(-1.0120)},
  {'text': 'Immunosuppressive drug', 'score': tensor(-1.0889)}],
 [{'text': 'Insulin', 'score': tensor(-0.3428)},
  {'text': 'Insulin-like growth factor', 'score': tensor(-0.3444)},
  {'text': 'Insulin receptor', 'score': tensor(-1.1370)},
  {'text': 'Insulin resistance', 'score': tensor(-1.5255)},
  {'text': 'Insulin receptor substrate', 'score': tensor(-2.6037)}]]